# Sandia / SNL extract notebook

This notebook is the Colab-side extract step for adding the Sandia National Labs / Preger dataset to the Dicle battery-transfer pipeline.

It does the heavy raw-data work in Drive/Colab, then downloads only compact reproducibility artifacts:

- `sandia_cycles_tidy.csv`
- `sandia_cell_metadata.csv`
- `sandia_cell_audit.csv`
- `sandia_threshold_summary.csv`
- `features_sop12_sandia.csv`
- optional `features_sop12_sandia_capnorm.csv`
- optional `features_sop12_combined_plus_sandia.csv`

The raw Sandia files are not copied into the repo and are not included in the ZIP.

## 0. Configuration

The default `SANDIA_ROOT` matches the folder used in the earlier exploratory notebook: `/content/drive/MyDrive/SandiaNationalLab/`.

In [ ]:
# Edit only if your Drive folder or branch name differs.
GITHUB_REPO = "https://github.com/osmansafacifci/Graduation-Project-Dicle.git"
BRANCH = "main"

SANDIA_ROOT = "/content/drive/MyDrive/SandiaNationalLab"
REPO_DIR = "/content/Graduation-Project-Dicle"

# Match the active thesis feature windows.
N_WINDOWS = [50, 100]
EOL_FRACTION = 0.85
EXTRA_EOL_FRACTIONS_FOR_AUDIT = [0.90, 0.85, 0.80]

# Raw Sandia time-series current convention: charge current > 0, discharge current < 0.
CHARGE_CURRENT_A = 0.2
DISCHARGE_CURRENT_A = -0.2

# The main benchmark remains raw capacity-only features. A cap-normalized copy is useful for ablations.
WRITE_CAPACITY_NORMALIZED_COPY = True
WRITE_COMBINED_PLUS_SANDIA = True

# If your Drive folder contains Google Sheets shortcuts instead of CSV files, set this True.
# CSV files are strongly preferred for speed.
ALLOW_GSHEET_SHORTCUTS = False

## 1. Clone the repo and install lightweight dependencies

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR)
if REPO_DIR.exists():
    print(f"[clone] repo already exists at {REPO_DIR}; fetching latest {BRANCH}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("[repo]", Path.cwd())

In [ ]:
%pip install -q numpy pandas scipy matplotlib

## 2. Mount Drive and discover Sandia time-series files

The notebook looks recursively under `SANDIA_ROOT` for files named like `*_timeseries.csv` or `*_timeseries.gsheet`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
SANDIA_ROOT = Path(SANDIA_ROOT)
if not SANDIA_ROOT.exists():
    raise FileNotFoundError(f"Sandia folder not found: {SANDIA_ROOT}")
print("[sandia root]", SANDIA_ROOT)

In [ ]:
import re
import pandas as pd

TIME_SUFFIXES = ["_timeseries.csv", "_timeseries", "_timeseries.gsheet"]

def strip_timeseries_suffix(name: str) -> str:
    for suffix in TIME_SUFFIXES:
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return Path(name).stem


def sanitize_cell_id(base: str) -> str:
    clean = re.sub(r"[^A-Za-z0-9]+", "_", base).strip("_")
    return f"sandia_{clean}"


def parse_sandia_metadata(base: str, path: Path) -> dict:
    # Earlier notebook used this approximate structure:
    # Institution Code / Form factor / Cathode / Temperature / Min-Max SOC / Charge-Discharge rate / Letter.
    parts = [p for p in re.split(r"[_\s]+", base) if p]
    keys = [
        "institution_code",
        "form_factor",
        "cathode",
        "temperature",
        "soc_window",
        "charge_discharge_rate",
        "replicate_letter",
    ]
    meta = {k: None for k in keys}
    for key, value in zip(keys, parts, strict=False):
        meta[key] = value
    meta.update(
        {
            "source_base": base,
            "source_file": str(path),
            "source_folder": str(path.parent),
            "cell_id": sanitize_cell_id(base),
        }
    )
    return meta


def discover_timeseries_files(root: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        name = path.name
        if not any(name.endswith(suffix) for suffix in TIME_SUFFIXES):
            continue
        base = strip_timeseries_suffix(name)
        rows.append(parse_sandia_metadata(base, path))
    files = pd.DataFrame(rows)
    if files.empty:
        raise FileNotFoundError(f"No *_timeseries.csv / *_timeseries.gsheet files found under {root}")

    # Prefer CSV over Google-Sheets shortcut if both exist for the same cell.
    files["is_csv"] = files["source_file"].str.endswith(".csv")
    files = (
        files.sort_values(["cell_id", "is_csv"], ascending=[True, False])
        .drop_duplicates("cell_id", keep="first")
        .drop(columns=["is_csv"])
        .reset_index(drop=True)
    )
    return files

sandia_files = discover_timeseries_files(SANDIA_ROOT)
print(f"[discover] {len(sandia_files)} unique Sandia time-series files")
display(sandia_files.head())
print(sandia_files["cathode"].value_counts(dropna=False).head(20))

## 3. Parse raw time-series files into a tidy per-cycle capacity table

For each cell and cycle, the notebook extracts:

- `Q_discharge`: max discharge capacity during discharge-current rows
- `Q_charge`: max charge capacity during charge-current rows
- voltage and point-count summaries

Cycle numbering is normalized to start at 1, so the downstream feature code matches the MATR/HUST convention.

In [ ]:
import json
import math
import tempfile
from subprocess import getoutput

import numpy as np
import pandas as pd

try:
    import gspread
    from google.auth import default as google_auth_default
except Exception:
    gspread = None

_GSPREAD_CLIENT = None

def maybe_gspread_client():
    global _GSPREAD_CLIENT
    if _GSPREAD_CLIENT is None:
        if gspread is None:
            raise RuntimeError("gspread is not available; install it or use CSV exports")
        from google.colab import auth
        auth.authenticate_user()
        creds, _ = google_auth_default()
        _GSPREAD_CLIENT = gspread.authorize(creds)
    return _GSPREAD_CLIENT


def read_gsheet_shortcut(path: Path) -> pd.DataFrame:
    if not ALLOW_GSHEET_SHORTCUTS:
        raise RuntimeError(
            f"Found Google Sheets shortcut but ALLOW_GSHEET_SHORTCUTS=False: {path}. "
            "Export as CSV or set ALLOW_GSHEET_SHORTCUTS=True."
        )
    subprocess.run(["apt-get", "install", "-y", "xattr"], check=True, stdout=subprocess.DEVNULL)
    file_id = getoutput(f"xattr -p 'user.drive.id' '{path}'").strip()
    if not file_id:
        raise RuntimeError(f"Could not read Drive file id from gsheet shortcut: {path}")
    sheet = maybe_gspread_client().open_by_key(file_id)
    worksheet = sheet.get_worksheet(0)
    values = worksheet.get_all_values()
    header = values.pop(0)
    return pd.DataFrame(values, columns=header)


def read_timeseries(path_like: str) -> pd.DataFrame:
    path = Path(path_like)
    if path.name.endswith(".gsheet"):
        return read_gsheet_shortcut(path)
    return pd.read_csv(path)


def normalized_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_to_col = {normalized_name(c): c for c in df.columns}
    for cand in candidates:
        hit = norm_to_col.get(normalized_name(cand))
        if hit is not None:
            return hit
    return None

COLUMN_CANDIDATES = {
    "cycle": ["Cycle_Index", "Cycle Index", "cycle", "Cycle", "Cycle_Index_"],
    "current": ["Current (A)", "Current", "current_a", "I", "Current_A"],
    "q_discharge": ["Discharge_Capacity (Ah)", "Discharge Capacity (Ah)", "Discharge_Capacity", "Q_discharge", "Qd"],
    "q_charge": ["Charge_Capacity (Ah)", "Charge Capacity (Ah)", "Charge_Capacity", "Q_charge", "Qc"],
    "voltage": ["Voltage (V)", "Voltage", "Voltage_V", "V"],
    "temperature": ["Temperature (C)", "Temperature", "Temperature_C", "Cell_Temperature (C)"],
}


def first_eol_cycle(qd: np.ndarray, q0: float, fraction: float) -> float:
    if not np.isfinite(q0) or q0 <= 0:
        return float("nan")
    threshold = fraction * q0
    for idx in range(1, len(qd)):
        q = qd[idx]
        if np.isfinite(q) and q > 0 and q <= threshold:
            return float(idx + 1)
    return float("nan")


def compute_q0(qd: np.ndarray) -> float:
    qd = np.asarray(qd, dtype=float).ravel()
    if len(qd) < 5:
        return float("nan")
    vals = qd[1:5]
    vals = vals[np.isfinite(vals) & (vals > 0)]
    return float(np.median(vals)) if len(vals) else float("nan")


def series_from_cycle_map(values: pd.Series, max_cycle: int) -> np.ndarray:
    indexed = pd.Series(index=np.arange(1, max_cycle + 1), dtype=float)
    indexed.loc[values.index.astype(int)] = pd.to_numeric(values, errors="coerce")
    indexed = indexed.interpolate(limit_direction="both")
    return indexed.to_numpy(dtype=float)


def parse_one_cell(meta: dict) -> tuple[pd.DataFrame | None, dict, np.ndarray | None]:
    path = meta["source_file"]
    raw = read_timeseries(path)
    if raw.empty:
        audit = {**meta, "parse_status": "empty_file"}
        return None, audit, None

    cols = {key: find_col(raw, choices) for key, choices in COLUMN_CANDIDATES.items()}
    required = ["cycle", "current", "q_discharge"]
    missing = [key for key in required if cols[key] is None]
    if missing:
        audit = {**meta, "parse_status": f"missing_columns:{','.join(missing)}", "columns": json.dumps(list(raw.columns))}
        return None, audit, None

    df = raw.copy()
    for col in [c for c in cols.values() if c is not None]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    cycle_raw = df[cols["cycle"]]
    df = df.loc[cycle_raw.notna()].copy()
    if df.empty:
        audit = {**meta, "parse_status": "no_valid_cycle_index"}
        return None, audit, None

    cycle_min = int(np.nanmin(df[cols["cycle"]].to_numpy(dtype=float)))
    if cycle_min <= 0:
        df["cycle_norm"] = df[cols["cycle"]].astype(int) + (1 - cycle_min)
    else:
        df["cycle_norm"] = df[cols["cycle"]].astype(int)
    max_cycle = int(df["cycle_norm"].max())

    current = df[cols["current"]]
    discharge = df.loc[current < DISCHARGE_CURRENT_A].copy()
    if discharge.empty:
        # Fallback for already-filtered discharge tables.
        discharge = df.loc[df[cols["q_discharge"]] > 0].copy()
    qd_by_cycle = discharge.groupby("cycle_norm")[cols["q_discharge"]].max()
    qd = series_from_cycle_map(qd_by_cycle, max_cycle)

    if cols["q_charge"] is not None:
        charge = df.loc[current > CHARGE_CURRENT_A].copy()
        qc_by_cycle = charge.groupby("cycle_norm")[cols["q_charge"]].max()
        qc = series_from_cycle_map(qc_by_cycle, max_cycle) if len(qc_by_cycle) else np.full(max_cycle, np.nan)
    else:
        qc = np.full(max_cycle, np.nan)

    cycle_rows = []
    grouped = df.groupby("cycle_norm")
    for cycle in range(1, max_cycle + 1):
        g = grouped.get_group(cycle) if cycle in grouped.groups else pd.DataFrame()
        row = {
            "cell_id": meta["cell_id"],
            "cycle": int(cycle),
            "Q_discharge": float(qd[cycle - 1]) if cycle - 1 < len(qd) else float("nan"),
            "Q_charge": float(qc[cycle - 1]) if cycle - 1 < len(qc) else float("nan"),
            "n_points": int(len(g)),
            "v_min": float(g[cols["voltage"]].min()) if cols["voltage"] and not g.empty else float("nan"),
            "v_max": float(g[cols["voltage"]].max()) if cols["voltage"] and not g.empty else float("nan"),
            "temperature_mean": float(g[cols["temperature"]].mean()) if cols["temperature"] and not g.empty else float("nan"),
        }
        cycle_rows.append(row)
    tidy = pd.DataFrame(cycle_rows)

    q0 = compute_q0(qd)
    life = first_eol_cycle(qd, q0, EOL_FRACTION)
    threshold_lives = {
        f"cycle_life_{str(frac).replace('.', 'p')}": first_eol_cycle(qd, q0, frac)
        for frac in EXTRA_EOL_FRACTIONS_FOR_AUDIT
    }
    audit = {
        **meta,
        "parse_status": "ok",
        "n_cycles": int(max_cycle),
        "q0": q0,
        "cycle_life": life,
        "is_censored": int(not np.isfinite(life)),
        "last_cycle": int(max_cycle),
        "last_qdis": float(qd[-1]) if len(qd) else float("nan"),
        "min_qdis": float(np.nanmin(qd)) if len(qd) else float("nan"),
        "missing_qdis_cycles": int(np.sum(~np.isfinite(qd))),
        "cycle_index_min_raw": cycle_min,
        "cycle_index_max_raw": int(np.nanmax(df[cols["cycle"]].to_numpy(dtype=float))),
        "columns": json.dumps(cols),
        **threshold_lives,
    }
    return tidy, audit, qd

In [ ]:
from collections import OrderedDict

all_tidy = []
audit_rows = []
qd_by_cell = OrderedDict()

for idx, meta in enumerate(sandia_files.to_dict("records"), start=1):
    print(f"[{idx:03d}/{len(sandia_files)}] {meta['cell_id']}")
    try:
        tidy, audit, qd = parse_one_cell(meta)
    except Exception as exc:
        tidy, qd = None, None
        audit = {**meta, "parse_status": f"error:{type(exc).__name__}", "error_message": str(exc)}
    audit_rows.append(audit)
    if tidy is not None and qd is not None:
        all_tidy.append(tidy)
        qd_by_cell[meta["cell_id"]] = qd

cycles_tidy = pd.concat(all_tidy, ignore_index=True) if all_tidy else pd.DataFrame()
cell_audit = pd.DataFrame(audit_rows)
cell_metadata = sandia_files.copy()

print("[parsed cells]", len(qd_by_cell))
print(cell_audit["parse_status"].value_counts(dropna=False))
display(cell_audit.head())

## 4. Save audit artifacts and build the 34 capacity-only feature table

The feature builder is imported directly from the repo so Sandia uses the same 34 feature definitions as MATR/HUST.

In [ ]:
import importlib.util

INTERMEDIATE_DIR = REPO_DIR / "data" / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

cycles_path = INTERMEDIATE_DIR / "sandia_cycles_tidy.csv"
audit_path = INTERMEDIATE_DIR / "sandia_cell_audit.csv"
metadata_path = INTERMEDIATE_DIR / "sandia_cell_metadata.csv"
threshold_path = INTERMEDIATE_DIR / "sandia_threshold_summary.csv"

cycles_tidy.to_csv(cycles_path, index=False)
cell_audit.to_csv(audit_path, index=False)
cell_metadata.to_csv(metadata_path, index=False)

summary_rows = []
for frac in EXTRA_EOL_FRACTIONS_FOR_AUDIT:
    col = f"cycle_life_{str(frac).replace('.', 'p')}"
    if col not in cell_audit.columns:
        continue
    ok = cell_audit[cell_audit["parse_status"] == "ok"].copy()
    reached = ok[col].notna() & np.isfinite(pd.to_numeric(ok[col], errors="coerce"))
    summary_rows.append(
        {
            "dataset": "sandia",
            "eol_fraction": frac,
            "n_cells_ok": int(len(ok)),
            "n_reached_eol": int(reached.sum()),
            "n_censored": int(len(ok) - reached.sum()),
            "reached_eol_fraction": float(reached.mean()) if len(ok) else float("nan"),
        }
    )
threshold_summary = pd.DataFrame(summary_rows)
threshold_summary.to_csv(threshold_path, index=False)

spec = importlib.util.spec_from_file_location("build_features", REPO_DIR / "1_features" / "build_features.py")
build_features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(build_features)

feature_rows = build_features.build_feature_rows(
    qd_by_cell,
    dataset="sandia",
    n_windows=tuple(N_WINDOWS),
    eol_fraction=EOL_FRACTION,
    capacity_normalize=False,
)
features_sandia = pd.DataFrame(feature_rows)
features_path = INTERMEDIATE_DIR / "features_sop12_sandia.csv"
features_sandia.to_csv(features_path, index=False)
print(f"[features] raw: {features_sandia.shape} -> {features_path}")

if WRITE_CAPACITY_NORMALIZED_COPY:
    cap_rows = build_features.build_feature_rows(
        qd_by_cell,
        dataset="sandia",
        n_windows=tuple(N_WINDOWS),
        eol_fraction=EOL_FRACTION,
        capacity_normalize=True,
    )
    features_capnorm = pd.DataFrame(cap_rows)
    capnorm_path = INTERMEDIATE_DIR / "features_sop12_sandia_capnorm.csv"
    features_capnorm.to_csv(capnorm_path, index=False)
    print(f"[features] capnorm: {features_capnorm.shape} -> {capnorm_path}")
else:
    capnorm_path = None

if WRITE_COMBINED_PLUS_SANDIA:
    combined_base_path = INTERMEDIATE_DIR / "features_sop12_combined.csv"
    if combined_base_path.exists():
        base = pd.read_csv(combined_base_path)
        combined = pd.concat([base, features_sandia], ignore_index=True, sort=False)
        combined_path = INTERMEDIATE_DIR / "features_sop12_combined_plus_sandia.csv"
        combined.to_csv(combined_path, index=False)
        print(f"[features] combined + sandia: {combined.shape} -> {combined_path}")
    else:
        combined_path = None
else:
    combined_path = None

display(threshold_summary)
if not features_sandia.empty:
    display(features_sandia.groupby(["dataset", "n_cycles", "is_censored"]).size().rename("n").reset_index())

## 5. Package compact outputs and download

This ZIP intentionally contains no raw Sandia files.

In [ ]:
import datetime
import zipfile
from google.colab import files

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
zip_path = Path("/content") / f"sandia_extract_outputs_{stamp}.zip"

include_paths = [
    cycles_path,
    audit_path,
    metadata_path,
    threshold_path,
    features_path,
]
if WRITE_CAPACITY_NORMALIZED_COPY and capnorm_path is not None:
    include_paths.append(capnorm_path)
if WRITE_COMBINED_PLUS_SANDIA and combined_path is not None:
    include_paths.append(combined_path)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in include_paths:
        if path is None or not Path(path).exists():
            continue
        zf.write(path, arcname=str(Path(path).relative_to(REPO_DIR)))
        print("[zip]", Path(path).relative_to(REPO_DIR))

print("[download]", zip_path)
files.download(str(zip_path))

## 6. Local next step

On your laptop, unzip the downloaded file into the repo root:

```bash
cd /Users/osmancifci/Downloads/Graduation-Project-Dicle
unzip -o ~/Downloads/sandia_extract_outputs_*.zip
```

Then commit the compact `data/intermediate/` files and run the local modeling/analysis extensions.